<a href="https://colab.research.google.com/github/vignanchintala/AI-Driven-Malaria-Diagnosis-Using-Compact-CNNs-on-Jetson-TX2/blob/main/gemma_chatbot_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🤖 Gemma 2B Chatbot - Google Colab
This notebook sets up a small chatbot using Google's `gemma-2b-it` model with Gradio UI. All dependencies are installed cleanly without PyTorch conflicts.

In [3]:
!pip uninstall -y torch torchvision torchaudio xformers fastai


Found existing installation: torch 2.6.0
Uninstalling torch-2.6.0:
  Successfully uninstalled torch-2.6.0
Found existing installation: torchvision 0.21.0
Uninstalling torchvision-0.21.0:
  Successfully uninstalled torchvision-0.21.0
Found existing installation: torchaudio 2.6.0
Uninstalling torchaudio-2.6.0:
  Successfully uninstalled torchaudio-2.6.0


In [4]:
!pip install torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0


  Using cached torch-2.6.0-cp311-cp311-manylinux1_x86_64.whl.metadata (28 kB)
  Using cached torchvision-0.21.0-cp311-cp311-manylinux1_x86_64.whl.metadata (6.1 kB)
  Using cached torchaudio-2.6.0-cp311-cp311-manylinux1_x86_64.whl.metadata (6.6 kB)
Using cached torch-2.6.0-cp311-cp311-manylinux1_x86_64.whl (766.7 MB)
Using cached torchvision-0.21.0-cp311-cp311-manylinux1_x86_64.whl (7.2 MB)
Using cached torchaudio-2.6.0-cp311-cp311-manylinux1_x86_64.whl (3.4 MB)


In [5]:
!pip install git+https://github.com/facebookresearch/xformers@v0.0.30#egg=xformers


  Cloning https://github.com/facebookresearch/xformers (to revision v0.0.30) to /tmp/pip-install-kuojl4hm/xformers_c3d189306fdf41538cc6ac7edfd14eeb
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/xformers /tmp/pip-install-kuojl4hm/xformers_c3d189306fdf41538cc6ac7edfd14eeb
  Running command git checkout -q 4cf69f0967128217f1798de70b3e4477de138570
  Resolved https://github.com/facebookresearch/xformers to commit 4cf69f0967128217f1798de70b3e4477de138570
  Running command git submodule update --init --recursive -q
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 821.2/821.2 MB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 101.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 77.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.7/897.7 kB 40.9 MB/s eta 0:00:00
  

In [6]:
!pip install transformers accelerate bitsandbytes gradio PyMuPDF -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 73.1 MB/s eta 0:00:00


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

model_id = "google/gemma-2b-it"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto", torch_dtype=torch.float16)

generator = pipeline("text-generation", model=model, tokenizer=tokenizer)


In [8]:
import fitz  # PyMuPDF

def extract_text_from_file(file):
    if file.name.endswith(".pdf"):
        doc = fitz.open(file.name)
        return "\n".join(page.get_text() for page in doc)
    elif file.name.endswith(".txt"):
        return file.read().decode("utf-8")
    else:
        return "Unsupported file format. Please upload PDF or TXT."


In [9]:
chat_history = []

def chat_with_gemma(user_input, file=None):
    global chat_history
    file_text = extract_text_from_file(file) if file else ""

    if file_text:
        user_input += f"\n\nReference Info:\n{file_text[:1000]}"

    formatted_history = ""
    for turn in chat_history[-3:]:
        formatted_history += f"<start_of_turn>user\n{turn['user']}<end_of_turn>\n"
        formatted_history += f"<start_of_turn>model\n{turn['bot']}<end_of_turn>\n"

    prompt = formatted_history + f"<start_of_turn>user\n{user_input}<end_of_turn>\n<start_of_turn>model\n"

    output = generator(prompt, max_new_tokens=256, do_sample=True, temperature=0.7)[0]["generated_text"]
    response = output.split("<start_of_turn>model\n")[-1].strip()

    chat_history.append({"user": user_input, "bot": response})
    return response


In [10]:
import gradio as gr

chat_ui = gr.ChatInterface(
    fn=chat_with_gemma,
    title="Gemma LLM Chatbot",
    additional_inputs=[gr.File(label="Upload a PDF or TXT", file_types=[".pdf", ".txt"])],
    description="Chat with Google's Gemma 2B model. Optionally upload a file for context."
)

chat_ui.launch(share=True)


/usr/local/lib/python3.11/dist-packages/gradio/chat_interface.py:339: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://be2784d2e55972318b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [11]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

model_id = "google/gemma-2b-it"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="cpu",
    torch_dtype=torch.float32
)

generator = pipeline("text-generation", model=model, tokenizer=tokenizer)


ImportError: cannot import name 'pipeline' from 'transformers' (/usr/local/lib/python3.11/dist-packages/transformers/__init__.py)

In [12]:
!pip uninstall -y transformers huggingface_hub
!pip install transformers==4.40.2 huggingface_hub -q

Found existing installation: transformers 4.52.4
Uninstalling transformers-4.52.4:
  Successfully uninstalled transformers-4.52.4
Found existing installation: huggingface-hub 0.33.0
Uninstalling huggingface-hub-0.33.0:
  Successfully uninstalled huggingface-hub-0.33.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.0/138.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 68.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.4/515.4 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 81.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 4.1.0 requires transformers<5.0.0,>=4.41.0, but you have transformers 4.40.2 which is incompatible.
